# Build the visits-level KPI table

In [1]:
## Load events table
import pandas as pd
df_events = pd.read_csv("events_kpi_table.csv", parse_dates=["date_time"])

In [2]:
df_events

,client_id,visitor_id,visit_id,process_step,date_time,step_num,previous_step_num,step_direction,time_on_each_step,next_step_num,jump_size
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,4,NaN,first_visit,NaN,4.0,0.0
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,4,4.0,repeat,52.0,NaN,NaN
2,9056452,306992881_89423906595,1000165_4190026492_760066,start,2017-06-04 01:07:29,0,NaN,first_visit,NaN,1.0,1.0
3,9056452,306992881_89423906595,1000165_4190026492_760066,step_1,2017-06-04 01:07:32,1,0.0,forward,3.0,2.0,1.0
4,9056452,306992881_89423906595,1000165_4190026492_760066,step_2,2017-06-04 01:07:56,2,1.0,forward,24.0,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
755400,7149380,483112224_46340533900,999992932_41666455053_671149,step_1,2017-06-06 15:46:24,1,0.0,forward,21.0,2.0,1.0
755401,7149380,483112224_46340533900,999992932_41666455053_671149,step_2,2017-06-06 15:47:32,2,1.0,forward,68.0,3.0,1.0
755402,7149380,483112224_46340533900,999992932_41666455053_671149,step_3,2017-06-06 16:01:46,3,2.0,forward,854.0,4.0,1.0
755403,7149380,483112224_46340533900,999992932_41666455053_671149,confirm,2017-06-06 16:04:08,4,3.0,forward,142.0,4.0,0.0


## Aggregate events into visits

One row per `(client_id, visit_id)`.

In [16]:
df_visits = df_events.groupby(["client_id", "visit_id"]).agg(
    visit_start = ("date_time", "min"),
    visit_end = ("date_time", "max"),
    max_step_reached = ("step_num", "max"),
    n_steps_session = ("step_num", "count"),
    had_skip = ("jump_size", lambda x: (x > 1).any()),
    had_repeat = ("jump_size", lambda x: (x == 0).any()),
    went_backwards = ("step_direction", lambda x: (x == "backward").any()),
    reached_confirm = ("step_num", lambda x: (x == 4).any())).reset_index()

In [21]:
df_visits['smooth_visit'] = (
    df_visits['reached_confirm']
    & ~df_visits['went_backwards']
    & ~df_visits['had_skip']
    & (df_visits['n_steps_session'] == 5)
)

df_visits['had_error'] = (
    df_visits['went_backwards']
    | df_visits['had_skip']
    | df_visits['had_repeat']
)

##  Derived KPIs

`error_rate` and `visit_duration_sec` are computed *after* the agg because they depend on columns the agg just created.


In [22]:
df_visits["visit_duration_sec"] = (df_visits["visit_end"] - df_visits["visit_start"]).dt.total_seconds()

np.int64(0)

##  Save

In [23]:
df_visits.to_csv("visits_kpi_table.csv", index=False)